In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder,PolynomialFeatures,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import VotingClassifier,GradientBoostingClassifier,RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline as pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [13]:
train_data = pd.read_csv('train_insurance.csv')
test_data = pd.read_csv('test_insurance.csv')
sample = pd.read_csv('sample_submission_insurance.csv')
train_data = train_data.drop(columns='policy_id')

In [15]:
to_convert_num = train_data.select_dtypes(['int64','float64'])
x0 = to_convert_num.drop(columns='is_claim')
y0 = to_convert_num['is_claim']

In [17]:
smote = SMOTE()
rus = RandomUnderSampler()
x1,y1 = smote.fit_resample(x0,y0)
x,y = rus.fit_resample(x1,y1)
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [20]:
to_convert_nums = to_convert_num.drop(columns='is_claim').columns
preprocess = ColumnTransformer(transformers=[('num',StandardScaler(),to_convert_nums)])

In [21]:
RFC = RandomForestClassifier(n_estimators=135,max_depth=9,max_leaf_nodes=27)
GBC = GradientBoostingClassifier(n_estimators=230)
XGBC = XGBClassifier(n_estimators=110)
LR = LogisticRegression(max_iter=30000)
estimators = [('lr',LR),('rfc',RFC),('gbc',GBC),('xgbc',XGBC)]
VC = VotingClassifier(estimators=estimators,voting='soft')

In [22]:
model = pipeline(steps=[('preprocessing',preprocess),('poly_features',PolynomialFeatures(degree=2)),('ensemble',VC)])
model.fit(x_train,y_train)

In [25]:
score = model.predict(x_test)
accuracy = accuracy_score(y_test, score)
accuracy

In [32]:
final_test = model.predict(test_data)
submission=pd.DataFrame({'policy_id':test_data['policy_id'],
                         'is_claim':final_test})

In [33]:
submission.to_csv('submission.csv',index=False)